# Stage 1 - Gemma 4 QA Fine-tuning

This notebook prepares the Colab runtime, installs dependencies, points at a local extracted training data directory inside the repository, and fine-tunes Gemma 4 with checkpoints and resume support.

In [1]:
#@title 1. Prepare repository on the Colab runtime
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Yongjooon/QA-AI-FineTunning.git"
REPO_NAME = "QA-AI-FineTunning"
REPO_DIR = Path(f"/content/{REPO_NAME}")
OVERRIDE_REPO_DIR = os.environ.get("QA_FINETUNE_REPO_DIR", "").strip()

if OVERRIDE_REPO_DIR and Path(OVERRIDE_REPO_DIR, "pyproject.toml").exists():
    REPO_DIR = Path(OVERRIDE_REPO_DIR)
    print(f"Using override repository directory: {REPO_DIR}")
elif (REPO_DIR / ".git").exists():
    print(f"Repository already exists. Updating: {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    if REPO_DIR.exists():
        print(f"Directory exists without git metadata. Removing: {REPO_DIR}")
        subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)
    print(f"Cloning latest repository from {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
repo_src = str(REPO_DIR / "src")
os.environ["PYTHONPATH"] = repo_src + os.pathsep + os.environ.get("PYTHONPATH", "")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

print("Current working directory:", os.getcwd())
print("Notebook file exists:", (REPO_DIR / "colab" / "01_train_gemma_qa.ipynb").exists())
print("Train script exists:", (REPO_DIR / "src" / "qafinetune" / "train.py").exists())
print("PYTHONPATH:", os.environ["PYTHONPATH"])


Cloning latest repository from https://github.com/Yongjooon/QA-AI-FineTunning.git
Current working directory: /content/QA-AI-FineTunning
Notebook file exists: True
Train script exists: True
PYTHONPATH: /content/QA-AI-FineTunning/src:/env/python


In [2]:
#@title 2. Install dependencies
%cd /content/QA-AI-FineTunning
%pip install -q -U pip
%pip uninstall -y transformers
%pip install -q -r requirements-colab.txt
%pip install -q -e . --no-deps

import qafinetune
import transformers
print("qafinetune import ok:", qafinetune.__file__)
print("transformers version:", transformers.__version__)


/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.1 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for qafinetune (pyproject.toml) ... done
qafinetune import ok: /content/QA-AI-FineTunning/src/qafinetune/__init__.py


In [3]:
#@title 3. Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#@title 4. Optional Hugging Face login if the model is gated
# from huggingface_hub import notebook_login
# notebook_login()

In [5]:
#@title 5. Training parameters
MODEL_NAME = "google/gemma-4-E2B-it"  # Gemma 4 model recommended for a T4 Colab runtime
PERSIST_ROOT = "/content/drive/MyDrive/qa_ai_finetuning"
RUN_NAME = "gemma4_qa_lora_run_01"
RESUME_MODE = "auto"  # auto | never | path
RESUME_CHECKPOINT = ""
NUM_TRAIN_EPOCHS = 3.0
LEARNING_RATE = 2e-4
SAVE_STEPS = 50
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 4
MAX_TRAIN_SAMPLES = 0

In [6]:
#@title 6. Inspect Colab runtime
import json
import sys
from pathlib import Path

repo_src = Path('/content/QA-AI-FineTunning/src')
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

from qafinetune.runtime import detect_runtime, suggest_training_preset

runtime_profile = detect_runtime()
print(json.dumps(runtime_profile, indent=2, ensure_ascii=False))
print("\nSuggested preset:")
try:
    print(json.dumps(suggest_training_preset(runtime_profile), indent=2, ensure_ascii=False))
except RuntimeError as exc:
    print(f"Preset unavailable yet: {exc}")
    print("Switch the Colab runtime to GPU and rerun this cell before training.")

{
  "python_version": "3.12.13",
  "platform": "Linux-6.6.113+-x86_64-with-glibc2.35",
  "torch_version": "2.10.0+cu128",
  "cuda_available": true,
  "cuda_version": "12.8",
  "gpu_count": 1,
  "gpu_name": "Tesla T4",
  "gpu_memory_gb": 14.56,
  "compute_capability": "7.5",
  "bf16_supported": true,
  "cwd": "/content/QA-AI-FineTunning"
}

Suggested preset:
{
  "per_device_train_batch_size": 1,
  "gradient_accumulation_steps": 16,
  "max_seq_length": 1536,
  "lora_rank": 16,
  "lora_alpha": 32,
  "bf16": true,
  "fp16": false
}


In [ ]:
#@title 7. Optional Smankusors Colab Monitor
ENABLE_COLAB_MONITOR = True  #@param {type:"boolean"}

if ENABLE_COLAB_MONITOR:
    from urllib.request import urlopen
    exec(urlopen("http://colab-monitor.smankusors.com/track.py").read())
    _colabMonitor = ColabMonitor().start()
else:
    print("Colab Monitor is disabled.")

In [ ]:
#@title 8. Select training source
from pathlib import Path

TRAIN_DATA_SOURCE = "local_dir"  #@param ["local_dir", "local_zip", "drive_path", "upload_widget", "gdown_file_id"]
LOCAL_TRAIN_DIR = "/content/QA-AI-FineTunning/data/playwright_gemma4_training_pack_v4_raw"  #@param {type:"string"}
LOCAL_TRAIN_ZIP_PATH = "/content/QA-AI-FineTunning/train_data.zip"  #@param {type:"string"}
DRIVE_TRAIN_ZIP_PATH = "/content/drive/MyDrive/train_data.zip"  #@param {type:"string"}
GDRIVE_FILE_ID = ""  #@param {type:"string"}
DOWNLOADED_TRAIN_ZIP_PATH = "/content/QA-AI-FineTunning/train_data.zip"

if TRAIN_DATA_SOURCE == "local_dir":
    TRAIN_SOURCE_PATH = LOCAL_TRAIN_DIR
elif TRAIN_DATA_SOURCE == "local_zip":
    TRAIN_SOURCE_PATH = LOCAL_TRAIN_ZIP_PATH
elif TRAIN_DATA_SOURCE == "drive_path":
    TRAIN_SOURCE_PATH = DRIVE_TRAIN_ZIP_PATH
elif TRAIN_DATA_SOURCE == "upload_widget":
    from google.colab import files
    uploaded = files.upload()
    TRAIN_SOURCE_PATH = next(iter(uploaded.keys()))
elif TRAIN_DATA_SOURCE == "gdown_file_id":
    if not GDRIVE_FILE_ID.strip():
        raise ValueError("GDRIVE_FILE_ID is empty.")
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    subprocess.run(["gdown", GDRIVE_FILE_ID.strip(), "-O", DOWNLOADED_TRAIN_ZIP_PATH], check=True)
    TRAIN_SOURCE_PATH = DOWNLOADED_TRAIN_ZIP_PATH
else:
    raise ValueError(f"Unsupported TRAIN_DATA_SOURCE: {TRAIN_DATA_SOURCE}")

train_source = Path(TRAIN_SOURCE_PATH)
if not train_source.exists():
    raise FileNotFoundError(f"Training source was not found: {train_source}")

print("Using training source:", train_source)
if train_source.is_dir():
    !find "$TRAIN_SOURCE_PATH" -maxdepth 2 -type f | sort
else:
    !ls -lh "$TRAIN_SOURCE_PATH"


In [ ]:
#@title 9. Run training
import os
import shlex
import subprocess
import sys
from pathlib import Path

REPO_DIR = "/content/QA-AI-FineTunning"
train_source_path = Path(TRAIN_SOURCE_PATH)
if not train_source_path.exists():
    raise FileNotFoundError(f"Training source was not found: {train_source_path}")

env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO_DIR}/src:" + env.get("PYTHONPATH", "")

command = [
    sys.executable, "-m", "qafinetune.train",
    "--model_name", MODEL_NAME,
    "--train_source", str(train_source_path),
    "--output_root", PERSIST_ROOT,
    "--run_name", RUN_NAME,
    "--resume_mode", RESUME_MODE,
    "--resume_checkpoint", RESUME_CHECKPOINT,
    "--num_train_epochs", str(NUM_TRAIN_EPOCHS),
    "--learning_rate", str(LEARNING_RATE),
    "--save_steps", str(SAVE_STEPS),
    "--logging_steps", str(LOGGING_STEPS),
    "--save_total_limit", str(SAVE_TOTAL_LIMIT),
    "--max_train_samples", str(MAX_TRAIN_SAMPLES),
]

print(" ".join(shlex.quote(x) for x in command))
print("PYTHONPATH =", env["PYTHONPATH"])
print("TRAIN_SOURCE_PATH =", train_source_path)

process = subprocess.Popen(
    command,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)

assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
print("\nRETURN CODE:", return_code)
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")


In [ ]:
#@title 10. Inspect saved run state
import json
from pathlib import Path

run_state_path = Path(PERSIST_ROOT) / "train_runs" / RUN_NAME / "run_state.json"
if not run_state_path.exists():
    raise FileNotFoundError(f"Run state file was not found: {run_state_path}")

print(run_state_path)
print(json.dumps(json.loads(run_state_path.read_text(encoding='utf-8')), indent=2, ensure_ascii=False))